# Trabajo Práctico 2 - Grupo 02

### Ensamble

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

In [30]:
# Deep learning y transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install transformers datasets accelerate

# NLP y preprocesamiento
!pip install spacy
!python -m spacy download es_core_news_sm

# ML clásico
!pip install scikit-learn xgboost joblib

# Data
!pip install pandas numpy scipy

# Visualización
!pip install matplotlib seaborn

# Análisis de sentimiento (SentiWordNet)
!pip install nltk

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 60.4 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [31]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/santiagomoreyra/common-ensamble")

NOMBRE_ENTREGA = "ensamble_rf_xgb_roberta_v1"

BASE = "/kaggle/input/datasets/santiagomoreyra/modelos-ensamble"

TRANSFORMER_DIRS = [
    f"{BASE}/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final",  # el nombre exacto que ves en la imagen
]

SKLEARN_PATHS = [
    f"{BASE}/rf_tfidf_select_v5.joblib",
    f"{BASE}/xgb_bow_aug_baseline_v6_3.joblib",
]

PESOS = [
    0.25,  # transformer
    0.25,  # random forest
    0.50,  # xgboost
]

assert len(PESOS) == len(TRANSFORMER_DIRS) + len(SKLEARN_PATHS), \
    "La cantidad de pesos debe ser igual a la cantidad total de modelos"
assert abs(sum(PESOS) - 1.0) < 1e-6, "Los pesos deben sumar 1.0"

print(f"Entrega: {NOMBRE_ENTREGA}")
print(f"Modelos transformer: {len(TRANSFORMER_DIRS)}")
print(f"Modelos sklearn:     {len(SKLEARN_PATHS)}")
print(f"Pesos: {PESOS}")

Entrega: ensamble_rf_xgb_roberta_v1
Modelos transformer: 1
Modelos sklearn:     2
Pesos: [0.25, 0.25, 0.5]


## 1. Imports

In [32]:
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score

from common.data_utils import get_split, make_bow, SEED
from common.preprocessing import clean_minimal, clean_classical
from common.evaluation import evaluate
from common.io_utils import make_submission

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
BATCH_SIZE = 64

print(f"Device: {DEVICE}")
np.random.seed(SEED)

Device: cuda


## 2. Carga de datos

In [33]:
train_df = pd.read_csv("/kaggle/input/datasets/santiagomoreyra/data-ensamble/data/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/santiagomoreyra/data-ensamble/data/test.csv")

X_train_raw, X_val_raw, y_train, y_val = get_split(train_df)

# Para transformers: clean_minimal
X_val_minimal  = np.array([clean_minimal(t) for t in X_val_raw])
X_test_minimal = np.array([clean_minimal(t) for t in test_df["text"].values])

# Para modelos sklearn: clean_classical (solo si hay SKLEARN_PATHS)
if SKLEARN_PATHS:
    print("Preprocesando para sklearn...")
    X_val_classical  = np.array([clean_classical(t) for t in X_val_raw])
    X_test_classical = np.array([clean_classical(t) for t in test_df["text"].values])
    print("Listo.")

print(f"Val: {len(X_val_minimal):,} | Test: {len(X_test_minimal):,}")

Preprocesando para sklearn...
Listo.
Val: 10,200 | Test: 8,500


## 3. Funciones de inferencia

In [34]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
    def __len__(self): return len(self.encodings["input_ids"])
    def __getitem__(self, idx): return {k: v[idx] for k, v in self.encodings.items()}


def get_transformer_probs(model_dir: str, texts: np.ndarray) -> np.ndarray:
    """Devuelve matriz (n, 3) de probabilidades softmax para un transformer guardado."""
    print(f"  Cargando {model_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model     = model.to(DEVICE)
    model.eval()

    dataset    = TextDataset(texts, tokenizer, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

    # Liberar VRAM antes de cargar el siguiente modelo
    del model
    torch.cuda.empty_cache()

    return np.vstack(all_probs)


def get_sklearn_probs(model_path: str, texts: np.ndarray) -> np.ndarray:
    """Devuelve matriz (n, 3) de probabilidades para un modelo sklearn."""
    print(f"  Cargando {model_path}...")
    model = joblib.load(model_path)
    return model.predict_proba(texts)


print("Funciones cargadas.")

Funciones cargadas.


## 4. Inferencia de cada modelo

In [35]:
import os
for d in TRANSFORMER_DIRS:
    print(d, "→", os.path.exists(d))

/kaggle/input/datasets/santiagomoreyra/modelos-ensamble/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final → True


In [36]:
import os
print(os.listdir("/kaggle/input/datasets/santiagomoreyra/modelos-ensamble"))

['rf_tfidf_select_v5.joblib', 'red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final', 'xgb_bow_aug_baseline_v6_3.joblib']


In [37]:
val_probs_list  = []
test_probs_list = []

# Transformers
for model_dir in TRANSFORMER_DIRS:
    print(f"\nTransformer: {model_dir}")
    val_probs_list.append(get_transformer_probs(model_dir, X_val_minimal))
    test_probs_list.append(get_transformer_probs(model_dir, X_test_minimal))

# Modelos sklearn
for i, model_path in enumerate(SKLEARN_PATHS):
    print(f"\nSklearn: {model_path}")
    val_probs_list.append(get_sklearn_probs(model_path, X_val_classical))
    test_probs_list.append(get_sklearn_probs(model_path, X_test_classical))

print(f"\nProbabilidades obtenidas de {len(val_probs_list)} modelos.")


Transformer: /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final
  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Sklearn: /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/rf_tfidf_select_v5.joblib
  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/rf_tfidf_select_v5.joblib...
  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/rf_tfidf_select_v5.joblib...

Sklearn: /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/xgb_bow_aug_baseline_v6_3.joblib
  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/xgb_bow_aug_baseline_v6_3.joblib...
  Cargando /kaggle/input/datasets/santiagomoreyra/modelos-ensamble/xgb_bow_aug_baseline_v6_3.joblib...

Probabilidades obtenidas de 3 modelos.


## 5. Ensamble por soft voting ponderado

In [38]:
# Promedio ponderado de probabilidades
val_probs_ensemble  = sum(w * p for w, p in zip(PESOS, val_probs_list))
test_probs_ensemble = sum(w * p for w, p in zip(PESOS, test_probs_list))

# Predicción final: clase con mayor probabilidad
y_pred_val  = np.argmax(val_probs_ensemble,  axis=1)
y_pred_test = np.argmax(test_probs_ensemble, axis=1)

# Evaluación en validación
evaluate(NOMBRE_ENTREGA, y_val, y_pred_val,
         hyperparams={
             "modelos": [Path(d).name for d in TRANSFORMER_DIRS] +
                        [Path(p).stem for p in SKLEARN_PATHS],
             "pesos": PESOS,
         })


=== ensamble_rf_xgb_roberta_v1 ===
Hiperparámetros: {'modelos': ['red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final', 'rf_tfidf_select_v5', 'xgb_bow_aug_baseline_v6_3'], 'pesos': [0.25, 0.25, 0.5]}

F1-macro:  0.8306
Precision: 0.8319
Recall:    0.8294
Accuracy:  0.8587

              precision    recall  f1-score   support

    negativa     0.8883    0.8924    0.8903      4080
      neutra     0.7035    0.6828    0.6930      2040
    positiva     0.9039    0.9130    0.9084      4080

    accuracy                         0.8587     10200
   macro avg     0.8319    0.8294    0.8306     10200
weighted avg     0.8576    0.8587    0.8581     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3641     344        95
neutra         346    1393       301
positiva       112     243      3725


## 6. Comparación contra modelos individuales

In [39]:
print("F1-macro por modelo individual vs ensamble:")
print(f"  Ensamble: {f1_score(y_val, y_pred_val, average='macro'):.4f}")
for i, (probs, peso) in enumerate(zip(val_probs_list, PESOS)):
    preds_i = np.argmax(probs, axis=1)
    f1_i    = f1_score(y_val, preds_i, average="macro")
    nombre  = (TRANSFORMER_DIRS + SKLEARN_PATHS)[i]
    print(f"  Modelo {i+1} (peso={peso}): {f1_i:.4f} — {Path(nombre).name}")

F1-macro por modelo individual vs ensamble:
  Ensamble: 0.8306
  Modelo 1 (peso=0.25): 0.7412 — red_neuronal_xlm_roberta_base_llm_lr2e5_ep3_final
  Modelo 2 (peso=0.25): 0.7499 — rf_tfidf_select_v5.joblib
  Modelo 3 (peso=0.5): 0.8395 — xgb_bow_aug_baseline_v6_3.joblib


## 7. Submission

In [44]:
Path("submissions").mkdir(exist_ok=True)

sub = pd.DataFrame({"id": test_df["id"].values, "label": y_pred_test.astype(int)})
sub.to_csv(f"submissions/submission_{NOMBRE_ENTREGA}.csv", index=False)

dist = sub["label"].value_counts(normalize=True).sort_index()
print(f"Guardado: submissions/submission_{NOMBRE_ENTREGA}.csv ({len(sub)} predicciones)")
print(f"Distribucion: {', '.join(f'clase {k}: {v:.1%}' for k, v in dist.items())}")

Guardado: submissions/submission_ensamble_rf_xgb_roberta_v1.csv (8500 predicciones)
Distribucion: clase 0: 41.3%, clase 1: 19.1%, clase 2: 39.6%
